In [1]:
import sys

sys.path.insert(0, "../src")

from dotenv import load_dotenv

load_dotenv("../.env.local")

from fsparts_scraper.crawler import (
    fetch_sitemap,
    parse_sitemap_xml,
    filter_product_urls,
    KNOWN_CATEGORY_SLUGS,
)
from fsparts_scraper.extractor import (
    fetch_product_html,
    extract_prices,
    extract_images,
    clean_html,
    close_browser,
)
from fsparts_scraper.llm import get_client, extract_product_data
import random, json

llm_client = get_client()
print("Groq client ready")

Groq client ready


In [2]:
xml = fetch_sitemap()
all_urls = parse_sitemap_xml(xml)
product_urls = filter_product_urls(all_urls, KNOWN_CATEGORY_SLUGS)
print(f"Total sitemap URLs: {len(all_urls)}")
print(f"Product URLs after filtering: {len(product_urls)}")
sample = random.sample(product_urls, min(5, len(product_urls)))
for u in sample:
    print(u)

Total sitemap URLs: 273
Product URLs after filtering: 208
https://www.fsparts.org/compresor-monofasico-danfoss-55hp-mt64-1vi-mt64hm1fve-para-r22
https://www.fsparts.org/-visores-de-liquido-78-conexion-soldar-indicador-de-humedad-
https://www.fsparts.org/filtro-secador-conexion-soldar-dk-082-14-refrigeracion-hvac
https://www.fsparts.org/compresor-rotativo-gmcc-12000-btu-r410a-pa118m1c-3dru
https://www.fsparts.org/valvula-de-retencioncheck-parker-dn20-paso-recto2


In [3]:
# Change index to inspect different products
url = sample[0]
print(f"Inspecting: {url}\n")

html = await fetch_product_html(url)
prices = extract_prices(html)
images = extract_images(html)
cleaned = clean_html(html)

print("=== PRICES ===")
print(prices)

print("\n=== IMAGES ===")
for img in images:
    print(img)

print("\n=== LLM EXTRACTION ===")
data = extract_product_data(cleaned, client=llm_client)
print(json.dumps(data, ensure_ascii=False, indent=2))

Inspecting: https://www.fsparts.org/compresor-monofasico-danfoss-55hp-mt64-1vi-mt64hm1fve-para-r22

=== PRICES ===
{'price_cop': 5589000, 'price_ws1': 5363000}

=== IMAGES ===
https://cdn.zyrosite.com/cdn-cgi/image/format=auto,w=768,fit=scale-down/cdn-ecommerce/store_01JHKJ9ADH4FX85C9GFWGA22S5/assets/83cd2f18-e156-4116-b81b-83868367bd91.jpg

=== LLM EXTRACTION ===
{
  "sku": "MT64-1VI",
  "name": "compresor monofásico danfoss 5.5hp mt64hm1fve",
  "description": "El compresor monofásico Danfoss MT64HM1FVE (MT64-1VI) es un compresor hermético reciprocante diseñado para aplicaciones de refrigeración de media temperatura con refrigerante R22",
  "brand_id": 5,
  "product_line_id": 3,
  "specs": [
    {
      "label": "Potencia",
      "value": "5.5HP"
    },
    {
      "label": "Refrigerante",
      "value": "R22"
    },
    {
      "label": "Fuente de alimentación",
      "value": "208-230/1/60"
    },
    {
      "label": "Tipo de compresor",
      "value": "Hermético reciprocante"
    

In [4]:
results = []
for url in sample:
    print(f"\n--- {url.split('/')[-1]} ---")
    html = await fetch_product_html(url)
    prices = extract_prices(html)
    images = extract_images(html)
    llm_data = extract_product_data(clean_html(html), client=llm_client)
    if llm_data:
        llm_data.update(prices)
        llm_data["image_urls_original"] = images
        llm_data["source_url"] = url
    results.append(
        {"url": url, "prices": prices, "images_count": len(images), "llm": llm_data}
    )
    print(
        f"  prices: {prices} | images: {len(images)} | sku: {llm_data.get('sku') if llm_data else 'FAILED'}"
    )

print(f"\n{sum(1 for r in results if r['llm'])} / {len(results)} extractions succeeded")


--- compresor-monofasico-danfoss-55hp-mt64-1vi-mt64hm1fve-para-r22 ---
  prices: {'price_cop': 5589000, 'price_ws1': 5363000} | images: 1 | sku: MT64-1VI

--- -visores-de-liquido-78-conexion-soldar-indicador-de-humedad- ---
  prices: {'price_cop': None, 'price_ws1': 97000} | images: 3 | sku: SGN 22s

--- filtro-secador-conexion-soldar-dk-082-14-refrigeracion-hvac ---
  prices: {'price_cop': 69900, 'price_ws1': 64990} | images: 5 | sku: DK-082S

--- compresor-rotativo-gmcc-12000-btu-r410a-pa118m1c-3dru ---
  prices: {'price_cop': None, 'price_ws1': 399000} | images: 1 | sku: PA118M1C-3DRU

--- valvula-de-retencioncheck-parker-dn20-paso-recto2 ---
  prices: {'price_cop': None, 'price_ws1': 2880000} | images: 1 | sku: None

5 / 5 extractions succeeded


In [5]:
from fsparts_scraper.extractor import close_browser

close_browser()